# uload — UniFast 利用率驱动 · 五类拓扑负载生成

仓库侧 `example/plugins.cpp`：usr 算法 33 个（扇出/扇入 1..10、反馈 1..10、src/sink/relay/burn/nop），
**cfg=每拍忙等 µs**。本 notebook 自包含。

## 五类 + 各自的入口 API（独立可配置参数都写在各函数 docstring）
| 类 | API | 结构 |
|---|---|---|
| 多速率 multihop | `gen_multihop(u,m,...)` | 多条独立多速率多跳链（paths×[src→relay^depth→sink]） |
| 多速率 fork | `gen_fork(u,m,...)` | 单源→扇出→各支链→多汇 |
| 多速率 join | `gen_join(u,m,...)` | 多源各支链→汇入→单汇 |
| 反馈 | `gen_feedback(u,m,...)` | 含 usr_acc 自反馈 loop |
| 混合 | `gen_mixed(u,m,...)` | 随机拼以上段 |

每类图内部都混：**周期任务 timed**（显式 `period`）+ **非周期任务 event**（不写 period、数据触发）。

## u 的正确构造（按“周期在超周期内的比例”定释放次数）
- 每节点先定**周期** $T_j$(ms，整除超周期 H)：
  timed：显式取 $T_j\in\{H,H/2,H/4\}$（多速率档）；
  event：继承前级周期，多前级取**最短**（最快）。
- 每超周期**释放次数**由比例定：
  $$n_j = H/T_j$$
- UniFast 摊每节点“周期占用比”$b_j$（即每次激活烧其自身周期的比例），$\sum_j b_j=u\cdot m$；
- 每次忙等（µs）：
  $$C_j = b_j\cdot T_j\cdot 1000 \;=\; b_j\,H\cdot1000/n_j$$
- 自检：$\sum_j n_jC_j/(mH\cdot1000)=\sum_j b_j/m=u$。

In [41]:
# ==== 库体：五类拓扑生成（每类独立 API，无聚合 build_*）====
import json, math, random, os
from math import gcd
import numpy as np, pandas as pd, matplotlib.pyplot as plt

VERSION="1.0.0"
FAN_OUT={1:"usr_relay",2:"usr_fork",3:"usr_fork3",4:"usr_fork4",5:"usr_fork5",6:"usr_fork6",
         7:"usr_fork7",8:"usr_fork8",9:"usr_fork9",10:"usr_fork10"}
FAN_IN={1:"usr_relay",2:"usr_join",3:"usr_join3",4:"usr_join4",5:"usr_join5",6:"usr_join6",
        7:"usr_join7",8:"usr_join8",9:"usr_join9",10:"usr_join10"}

def unifast(rng,n,total,umax=1.0):
    for _ in range(200000):
        vals,s=[],total
        for i in range(n-1):
            nxt=s*(rng.random()**(1.0/(n-i))); vals.append(s-nxt); s=nxt
        vals.append(s)
        if all(0.0<=v<=umax for v in vals): return vals
    raise RuntimeError(f"UniFast 无法分配 n={n}, total={total:.3f}（任务数太少/降 u）")

def save(path,nodes):
    with open(path,"w") as f: json.dump(nodes,f,indent=1)

# ---- 骨架（节点：algo/fo/ins=[(前驱idx,其第几路输出)]/loop=?N）----
def _add(sk,algo,fo,ins=None,loop=None):
    sk.append({"algo":algo,"fo":fo,"loop":loop})
    if ins: sk[-1]["ins"]=ins
    return len(sk)-1

def _sk_multihop(paths,depth):
    sk=[]
    for _ in range(paths):
        r=_add(sk,"usr_src",1)
        for __ in range(depth): r=_add(sk,"usr_relay",1,[(r,0)])
        _add(sk,"usr_sink",1,[(r,0)])
    return sk

def _sk_fork(fan,bdepth):
    sk=[]; r=_add(sk,"usr_src",1); f=_add(sk,FAN_OUT[fan],fan,[(r,0)])
    for b in range(fan):
        prev=f
        for __ in range(bdepth): prev=_add(sk,"usr_relay",1,[(prev,0)])
        _add(sk,"usr_sink",1,[(prev,0)])
    return sk

def _sk_join(fan,bdepth,tail=2):
    sk=[]; lane=[]
    for _ in range(fan):
        r=_add(sk,"usr_src",1); prev=r
        for __ in range(bdepth): prev=_add(sk,"usr_relay",1,[(prev,0)])
        lane.append((prev,0))
    j=_add(sk,FAN_IN[fan],1,lane); prev=j
    for __ in range(tail): prev=_add(sk,"usr_relay",1,[(prev,0)])
    _add(sk,"usr_sink",1,[(prev,0)]); return sk

def _sk_feedback(depth,loop_pos,loopN):
    sk=[]; r=_add(sk,"usr_src",1)
    for i in range(depth):
        r=_add(sk,"usr_acc",1,[(r,0)],loop=loopN) if i in loop_pos else _add(sk,"usr_relay",1,[(r,0)])
    _add(sk,"usr_sink",1,[(r,0)]); return sk

def _sk_mixed(rng,nseg):
    sk=[]; r=_add(sk,"usr_src",1)
    for _ in range(nseg):
        x=rng.random()
        if x<0.45:
            for __ in range(rng.randint(1,3)): r=_add(sk,"usr_relay",1,[(r,0)])
        elif x<0.75:
            fan=rng.randint(2,5); f=_add(sk,FAN_OUT[fan],fan,[(r,0)]); tails=[]
            for b in range(fan):
                prev=f
                for __ in range(rng.randint(0,1)): prev=_add(sk,"usr_relay",1,[(prev,0)])
                tails.append((prev,0))
            r=_add(sk,FAN_IN[fan],1,tails)
        else:
            r=_add(sk,"usr_acc",1,[(r,0)],loop=rng.randint(2,6))
    _add(sk,"usr_sink",1,[(r,0)]); return sk

def _trig_emit(sk,u,m,H_ms,seed,ptimed=0.35):
    """定周期→释放次数→分预算→反算忙等（u 逻辑见 md 公式）。
    返回 (nodes, stats)；stats 含 per-node T/n 便于核对。"""
    rng=random.Random(seed); N=len(sk)
    pred=[[pi for (pi,_) in nd.get("ins",[])] for nd in sk]
    indeg=[len(p) for p in pred]; from collections import deque
    dq=deque(i for i in range(N) if indeg[i]==0); order=[]
    while dq:
        uu=dq.popleft(); order.append(uu)
        for v,pv in enumerate(pred):
            if uu in pv:
                indeg[v]-=1
                if indeg[v]==0: dq.append(v)
    for i in range(N):
        if i not in order: order.append(i)
    T=[0]*N; trig=[None]*N
    RATES=[H_ms, H_ms//2, H_ms//4]            # 周期档：H, H/2, H/4（多速率；须整除 H）
    for i in order:
        ins=sk[i].get("ins",[])
        if not ins:                            # 源：周期任务
            trig[i]="timed"; T[i]=int(rng.choice(RATES))
        elif rng.random()<ptimed:              # 中间节点随机为周期任务（显式 period，独立定时读最新）
            trig[i]="timed"; T[i]=int(rng.choice(RATES))
        else:                                  # 事件任务：继承前级周期，多前级取最短(最快)
            trig[i]="event"; T[i]=min(T[pi] for (pi,_) in ins) if ins else H_ms
    n=[H_ms//T[i] for i in range(N)]           # 释放次数=其在超周期内的比例
    assert all(H_ms%T[i]==0 and n[i]>=1 for i in range(N))
    b=unifast(random.Random(seed+1000),N,u*m,umax=1.0)   # Σb=u·m（每节点周期占用比）
    spins=[round(b[i]*T[i]*1000) for i in range(N)]      # C_j=b_j·T_j·1000 µs
    outs=[]
    for i,nd in enumerate(sk): outs.append([f"p{i}_{k}" for k in range(nd["fo"])])
    nodes=[]; realized=0.0
    for i,nd in enumerate(sk):
        ext=[outs[pi][pk] for (pi,pk) in nd.get("ins",[])]
        entry={"id":f"n{i}","name":nd["algo"],"version":VERSION,"outputs":outs[i],
               "wcet":round(spins[i]/1000.0,3),"parameters":[{"value":int(spins[i])}]}
        if trig[i]=="timed": entry["period"]=T[i]
        if nd.get("loop") is not None:
            o=outs[i][0]; ext.append(o)
            entry["loop"]={o:nd["loop"] if isinstance(nd["loop"],int) else 4}
        if ext: entry["inputs"]=ext
        nodes.append(entry); realized+=n[i]*spins[i]/(H_ms*1000.0)
    return nodes, {"nodes":N,"realized_u":realized/m,"n_max":max(n),"timed":trig.count("timed"),
                   "event":trig.count("event")}

def gen_multihop(u, m, H_ms=1000, seed=1, paths=None, depth=None, ptimed=0.35):
    """多速率 multihop：多条独立多跳链(src→relay^depth→sink)。
    可配置：paths(链数,默认由 seed 在[1..4]随机)、depth(每链跳,默认[2..6]随机)、
    ptimed(中间节点为周期任务的概率)；u/m/H_ms/seed 公用。返回 (nodes,stats)。"""
    rng=random.Random(seed)
    paths = rng.randint(1,4) if paths is None else paths
    depth = rng.randint(2,6) if depth is None else depth
    return _trig_emit(_sk_multihop(paths,depth),u,m,H_ms,seed,ptimed)

def gen_fork(u, m, H_ms=1000, seed=1, fan=None, bdepth=None, ptimed=0.35):
    """多速率 fork：单源→usr_fork(fan)→fan 支链(relay^bdepth)→多汇。
    可配置：fan(扇出度,默认由 seed 在[2..8]随机)、bdepth(每支链深,默认[1..3]随机)、ptimed。
    返回 (nodes,stats)。"""
    rng=random.Random(seed)
    fan = rng.randint(2,8) if fan is None else fan
    bdepth = rng.randint(1,3) if bdepth is None else bdepth
    return _trig_emit(_sk_fork(fan,bdepth),u,m,H_ms,seed,ptimed)

def gen_join(u, m, H_ms=1000, seed=1, fan=None, bdepth=None, tail=None, ptimed=0.35):
    """多速率 join：fan 个源各支链(relay^bdepth)→usr_join(fan)→tail 段→单汇。
    可配置：fan(扇入度,默认[2..8])、bdepth(支链深,默认[1..3])、tail(汇后链长,默认[1..3])、ptimed。"""
    rng=random.Random(seed)
    fan = rng.randint(2,8) if fan is None else fan
    bdepth = rng.randint(1,3) if bdepth is None else bdepth
    tail = rng.randint(1,3) if tail is None else tail
    return _trig_emit(_sk_join(fan,bdepth,tail),u,m,H_ms,seed,ptimed)

def gen_feedback(u, m, H_ms=1000, seed=1, depth=None, loopN=None, loop_pos=None, ptimed=0.35):
    """反馈：src→…(acc 插在某 hop,loop 窗口 loopN)…→sink。
    可配置：depth(链长,默认[3..8])、loopN(自反馈窗口,默认[2..8])、loop_pos(acc 位置,默认随机)、ptimed。"""
    rng=random.Random(seed)
    depth = rng.randint(3,8) if depth is None else depth
    loopN = rng.randint(2,8) if loopN is None else loopN
    pos = loop_pos if loop_pos is not None else [rng.randint(1,max(1,depth-1))]
    return _trig_emit(_sk_feedback(depth,pos,loopN),u,m,H_ms,seed,ptimed)

def gen_mixed(u, m, H_ms=1000, seed=1, nseg=None, ptimed=0.35):
    """混合：随机拼 chain/fork→join/反馈 段成单 DAG。
    可配置：nseg(随机段数,默认[3..8])、ptimed；fan/深/loop 窗口全由 seed 随机。"""
    rng=random.Random(seed)
    nseg = rng.randint(3,8) if nseg is None else nseg
    return _trig_emit(_sk_mixed(rng,nseg),u,m,H_ms,seed,ptimed)

# 通用：为某一类批量生成随机实例（每张独立随机 seed，写文件+本类 manifest）
def batch_kind(kind, gen,
               out_dir='pipeline/uload_r',
               utils=(0.3, 0.6), workers=(1, 2, 4),
               n_per=2, H_ms=1000, seed_base=None):
    import csv as _c, random as _r
    os.makedirs(out_dir, exist_ok=True)
    if seed_base is None: seed_base = _r.randrange(1_000_000_000)
    rows, i = [], 0
    for u in utils:
        for m in workers:
            for _ in range(n_per):
                seed = seed_base + i; i += 1
                nodes, st = gen(u=u, m=m, H_ms=H_ms, seed=seed)
                fn = f"cfg_{kind}_u{int(round(u*100))}_m{m}_s{seed}.json"
                save(os.path.join(out_dir, fn), nodes)
                rows.append({"kind": kind, "u": u, "m": m, "seed": seed, "file": fn,
                             "nodes": st["nodes"], "timed": st["timed"],
                             "event": st["event"], "realized_u": st["realized_u"]})
    with open(os.path.join(out_dir, f"manifest_{kind}.csv"), "w", newline="") as fp:
        w = _c.DictWriter(fp, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    return rows


### 五种类型批量代码段（每类一段，独立随机 seed 并记录）
每段调用库内 `batch_kind(kind, gen, out_dir, utils, workers, n_per, H_ms, seed_base)`：
该类按 u×m×n_per 随机生成 `n_per` 份/格点，每张独立 seed 写 `cfg_<kind>_..._s<seed>.json`
+ 该类自己的 `manifest_<kind>.csv`；`seed_base=None` 自动随机，想复现传基数。

#### 多速率多线 multihop

In [42]:
# 多速率多线 multihop：批量随机
r_multihop = batch_kind(
    'multihop',
    gen_multihop,
    'uload/multihop',
    utils=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
    workers=[1, 2, 3, 4, 5, 6],
    n_per=1, seed_base=10_000)
print('multihop:', len(r_multihop), '张; 命中u:', all(abs(x['realized_u']-x['u'])<1e-3 for x in r_multihop))


multihop: 54 张; 命中u: True


#### 多速率 fork

In [43]:
# 多速率 fork：批量随机
r_fork = batch_kind(
    'fork',
    gen_fork,
    'uload/fork',
    utils=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
    workers=[1, 2, 3, 4, 5, 6],
    n_per=1, seed_base=10_000)
print('fork:', len(r_fork), '张; 命中u:', all(abs(x['realized_u']-x['u'])<1e-3 for x in r_fork))


fork: 54 张; 命中u: True


#### 多速率 join

In [44]:
# 多速率 join：批量随机
r_join = batch_kind(
    'join',
    gen_join,
    'uload/join',
    utils=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
    workers=[1, 2, 3, 4, 5, 6],
    n_per=1, seed_base=10_000)
print('join:', len(r_join), '张; 命中u:', all(abs(x['realized_u']-x['u'])<1e-3 for x in r_join))


join: 54 张; 命中u: True


#### 反馈 loop(=gen_feedback)

In [45]:
# 反馈 loop(=gen_feedback)：批量随机
r_loop = batch_kind(
    'loop',
    gen_feedback,
    'uload/loop',
    utils=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
    workers=[1, 2, 3, 4, 5, 6],
    n_per=1, seed_base=10_000)
print('loop:', len(r_loop), '张; 命中u:', all(abs(x['realized_u']-x['u'])<1e-3 for x in r_loop))


loop: 54 张; 命中u: True


#### 混合 mixed

In [46]:
# 混合 mixed：批量随机
r_mixed = batch_kind(
    'mixed',
    gen_mixed,
    'uload/mixed',
    utils=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
    workers=[1, 2, 3, 4, 5, 6],
    n_per=1, seed_base=10_000)
print('mixed:', len(r_mixed), '张; 命中u:', all(abs(x['realized_u']-x['u'])<1e-3 for x in r_mixed))


mixed: 54 张; 命中u: True
